In [8]:
import os
import re
import zipfile
import xml.etree.ElementTree as ET
import pandas as pd
from collections import defaultdict

from sympy import (
    symbols, Poly, fraction, together, simplify, Pow, log,
    Add, Abs, sin, cos, tan, sqrt, Dummy
)
from sympy.parsing.sympy_parser import (
    parse_expr,
    standard_transformations,
    implicit_multiplication_application,
    convert_xor
)

# Folder path and output file
folder_path = "/Users/guillermobautista/desktop/JKU/Function Arts/DataAnalysis/Sample3"
output_file = os.path.join(folder_path, "sample3.csv")

x = symbols("x")
transformations = standard_transformations + (
    implicit_multiplication_application,
    convert_xor
)

# --- Helper functions ---

def extract_parenthesized(text, open_idx):
    """
    Given the index of '(' in text, return:
        (inside_text, closing_paren_index)
    """
    depth = 0
    for i in range(open_idx, len(text)):
        if text[i] == '(':
            depth += 1
        elif text[i] == ')':
            depth -= 1
            if depth == 0:
                return text[open_idx + 1:i], i
    raise ValueError("Unmatched parenthesis")


def split_top_level_args(s):
    """
    Split a string by commas at top level only.
    """
    parts = []
    depth = 0
    current = []

    for ch in s:
        if ch == '(':
            depth += 1
            current.append(ch)
        elif ch == ')':
            depth -= 1
            current.append(ch)
        elif ch == ',' and depth == 0:
            parts.append(''.join(current).strip())
            current = []
        else:
            current.append(ch)

    parts.append(''.join(current).strip())
    return parts


def normalize_inequality_symbols(s):
    return (
        s.replace("≤", "<=")
         .replace("≥", ">=")
         .replace("≦", "<=")
         .replace("≧", ">=")
    )


def is_domain_restriction(part):
    s = normalize_inequality_symbols(part.strip())

    if s.startswith("(") and s.endswith(")"):
        s = s[1:-1].strip()

    if "x" not in s:
        return False
    if not any(op in s for op in ["<", ">", "<=", ">="]):
        return False

    forbidden_formula_tokens = ["sin", "cos", "tan", "log", "ln", "lg", "sqrt", "^", "*", "/", "Abs", "abs"]
    if any(tok in s for tok in forbidden_formula_tokens):
        return False

    chained_patterns = [
        r'^.+?(<=|<)\s*x\s*(<=|<)\s*.+$',
        r'^.+?(>=|>)\s*x\s*(>=|>)\s*.+$',
        r'^x\s*(<=|<)\s*.+$',
        r'^x\s*(>=|>)\s*.+$',
        r'^.+\s*(<=|<)\s*x$',
        r'^.+\s*(>=|>)\s*x$'
    ]

    return any(re.fullmatch(p, s) for p in chained_patterns)


def strip_trailing_domain_restriction(expr):
    parts = split_top_level_args(expr)
    if len(parts) >= 2:
        last_part = parts[-1].strip()
        if is_domain_restriction(last_part):
            return ','.join(parts[:-1]).strip()
    return expr.strip()


def strip_boolean_interval_factor(expr):
    s = expr.strip()

    changed = True
    while changed and s.startswith("(") and s.endswith(")"):
        changed = False
        depth = 0
        wraps_all = True
        for i, ch in enumerate(s):
            if ch == "(":
                depth += 1
            elif ch == ")":
                depth -= 1
                if depth == 0 and i != len(s) - 1:
                    wraps_all = False
                    break
        if wraps_all:
            s = s[1:-1].strip()
            changed = True

    factors = []
    depth = 0
    current = []
    for ch in s:
        if ch == "(":
            depth += 1
            current.append(ch)
        elif ch == ")":
            depth -= 1
            current.append(ch)
        elif ch == "*" and depth == 0:
            factors.append(''.join(current).strip())
            current = []
        else:
            current.append(ch)
    factors.append(''.join(current).strip())

    if len(factors) != 2:
        return expr.strip()

    f1, f2 = factors[0], factors[1]

    if is_domain_restriction(f1):
        return f2
    if is_domain_restriction(f2):
        return f1

    return expr.strip()


def clean_expression(expr):
    expr = expr.strip()

    if ":" in expr:
        left, right = expr.split(":", 1)
        if "=" in right:
            expr = right.strip()

    match = re.match(r'\s*\w+\(x\)\s*=\s*If\[[^\]]+\s*,\s*(.+?)\s*\]\s*$', expr)
    if match:
        rhs = match.group(1).strip()
        rhs = strip_trailing_domain_restriction(rhs)
        rhs = strip_boolean_interval_factor(rhs)
        return rhs

    match = re.match(r'\s*\w+\(x\)\s*=\s*(.+)$', expr)
    if match:
        rhs = match.group(1).strip()
        rhs = strip_trailing_domain_restriction(rhs)
        rhs = strip_boolean_interval_factor(rhs)
        return rhs

    match = re.match(r'\s*y\s*=\s*(.+)$', expr, flags=re.IGNORECASE)
    if match:
        rhs = match.group(1).strip()
        rhs = strip_trailing_domain_restriction(rhs)
        rhs = strip_boolean_interval_factor(rhs)
        return rhs

    if "=" in expr:
        parts = expr.split("=", 1)
        rhs = parts[1].strip()
        rhs = strip_trailing_domain_restriction(rhs)
        rhs = strip_boolean_interval_factor(rhs)
        return rhs

    expr = strip_trailing_domain_restriction(expr)
    expr = strip_boolean_interval_factor(expr)
    return expr


def replace_log_with_base_underscore(expr):
    """
    Convert:
        log_10(x)     -> log(x,10)
        log_{10}(x)   -> log(x,10)
        log_(10)(x)   -> log(x,10)
    """
    pattern1 = re.compile(r'log_\{\s*([^{}]+?)\s*\}\s*\(')
    pattern2 = re.compile(r'log_([A-Za-z0-9\.\+\-\*/]+)\s*\(')
    pattern3 = re.compile(r'log_\(\s*([^\(\)]+?)\s*\)\s*\(')

    while True:
        m = pattern1.search(expr)
        if not m:
            break
        base = m.group(1).strip()
        start = m.start()
        open_paren = m.end() - 1
        arg, close_idx = extract_parenthesized(expr, open_paren)
        replacement = f'log({arg},{base})'
        expr = expr[:start] + replacement + expr[close_idx + 1:]

    while True:
        m = pattern3.search(expr)
        if not m:
            break
        base = m.group(1).strip()
        start = m.start()
        open_paren = m.end() - 1
        arg, close_idx = extract_parenthesized(expr, open_paren)
        replacement = f'log({arg},{base})'
        expr = expr[:start] + replacement + expr[close_idx + 1:]

    while True:
        m = pattern2.search(expr)
        if not m:
            break
        base = m.group(1).strip()
        start = m.start()
        open_paren = m.end() - 1
        arg, close_idx = extract_parenthesized(expr, open_paren)
        replacement = f'log({arg},{base})'
        expr = expr[:start] + replacement + expr[close_idx + 1:]

    return expr


def looks_like_numeric_constant(s):
    s = s.strip()
    if not s:
        return False
    if "x" in s.lower():
        return False
    return bool(re.fullmatch(r'[-+]?\d*\.?\d+([eE][-+]?\d+)?', s))


def replace_log_two_argument_form(expr):
    """
    Convert only GeoGebra-style two-argument log:
        log(10, x)   -> log(x,10)
        log(2, x+1)  -> log(x+1,2)

    Leave canonical SymPy-style forms unchanged:
        log(x,10)
    """
    i = 0
    result = []

    while i < len(expr):
        if expr[i:i+4].lower() == 'log(':
            open_idx = i + 3
            inside, close_idx = extract_parenthesized(expr, open_idx)
            parts = split_top_level_args(inside)

            if len(parts) == 2:
                first, second = parts[0].strip(), parts[1].strip()

                if looks_like_numeric_constant(first) and "x" in second.lower():
                    result.append(f'log({second},{first})')
                else:
                    result.append(f'log({inside})')
            else:
                result.append(f'log({inside})')

            i = close_idx + 1
        else:
            result.append(expr[i])
            i += 1

    return ''.join(result)


def preprocess_for_sympy(expr):
    expr = expr.strip()

    # FIX: convert GeoGebra's lg( (log base 10) to log( before any other processing
    expr = re.sub(r'\blg\s*\(', 'log(', expr, flags=re.IGNORECASE)

    expr = re.sub(r'\bln\s*\(', 'log(', expr, flags=re.IGNORECASE)
    expr = re.sub(r'\babs\s*\(', 'Abs(', expr, flags=re.IGNORECASE)

    expr = replace_log_with_base_underscore(expr)
    expr = replace_log_two_argument_form(expr)

    return expr


def replace_log_calls_with_placeholder(expr, placeholder="L"):
    """
    Replace every full log(...) call (with balanced parentheses) by a placeholder.
    Works on preprocessed strings where log notation has already been normalized.
    """
    result = []
    i = 0

    while i < len(expr):
        if expr[i:i+4].lower() == 'log(':
            open_idx = i + 3
            _, close_idx = extract_parenthesized(expr, open_idx)
            result.append(placeholder)
            i = close_idx + 1
        else:
            result.append(expr[i])
            i += 1

    return ''.join(result)


def is_pure_log_expression_str(expr):
    """
    String-level detector for logarithmic expressions.
    After preprocessing (which converts lg, ln -> log), replace every log(...)
    block by a placeholder L. If no raw x remains outside those blocks,
    classify as LOG.

    Accepts:
        log_10(x)
        (log_10(x+2))^2
        (log_10(2x-6))^3
        ln(x)
        ln(x)^2
        lg(x)
        (lg(x+2))^2
        (lg(x+3))^3

    Rejects:
        x + log_10(x)
        x*log_10(x)
        log_10(x)/x
    """
    # Preprocess first so lg( and ln( become log(
    expr_pre = preprocess_for_sympy(expr)
    expr_pre = expr_pre.replace(" ", "")

    # If there is no log at all, it is not logarithmic
    if "log(" not in expr_pre.lower():
        return False

    # Exclude other special families for plain LOG classification
    lower = expr_pre.lower()
    forbidden = ["sin(", "cos(", "tan(", "sqrt(", "abs(", "exp("]
    if any(tok in lower for tok in forbidden):
        return False

    replaced = replace_log_calls_with_placeholder(expr_pre, "L")

    # Remove numeric constants, operators, placeholders, parentheses, commas, powers
    stripped = re.sub(r'[-+]?\d*\.?\d+([eE][-+]?\d+)?', '', replaced)
    stripped = stripped.replace("L", "")
    stripped = re.sub(r'[\+\-\*/\^\(\),]', '', stripped)

    # If x remains outside the log blocks, it is not a pure log family
    return "x" not in stripped.lower()


def remove_constant_addition(expr_sym):
    var_terms = [t for t in Add.make_args(expr_sym) if t.has(x)]
    if not var_terms:
        return 0
    return simplify(Add(*var_terms))


def split_scalar_and_core(expr_sym):
    coeff, rest = expr_sym.as_independent(x, as_Add=False)
    return simplify(coeff), simplify(rest)


# --- Family detectors ---

def is_constant_function(expr_sym):
    return not expr_sym.has(x)


def is_affine_in_x(expr):
    try:
        expr = simplify(expr)
        poly = Poly(expr, x)
        return poly.degree() <= 1
    except Exception:
        return False


def is_exponential_core(core):
    rewritten = simplify(core.rewrite(Pow))

    if isinstance(rewritten, Pow):
        base, exponent = rewritten.args

        if base.has(x):
            return False
        if not exponent.has(x):
            return False
        if not is_affine_in_x(exponent):
            return False

        return True

    return False


def is_logarithmic_core(core):
    """
    SymPy-level detector for logarithmic expressions.

    Handles:
        log(x+10)        — plain log
        log(x+10)**2     — Pow whose base is a log (e.g. from (lg(x+2))^2)
        log(x+10)**3     — higher integer powers of log
        2*log(x+10)      — scalar multiple (split_scalar_and_core handles upstream)
    """
    core = simplify(core)

    if core.has(sin) or core.has(cos) or core.has(tan) or core.has(Abs):
        return False

    # Handle Pow(log(...), n) — e.g. log(x+2)**2 or log(x+3)**3
    if isinstance(core, Pow):
        base, exponent = core.args
        if not exponent.has(x) and isinstance(base, log) and base.args[0].has(x):
            return True

    log_atoms = [a for a in core.atoms(log) if len(a.args) >= 1 and a.args[0].has(x)]

    if not log_atoms:
        return False

    replacements = {}
    for i, atom in enumerate(log_atoms):
        replacements[atom] = Dummy(f"L{i}")
    replaced = core.xreplace(replacements)

    return not replaced.has(x)


def is_sine_core(core):
    return core.func == sin and core.args and core.args[0].has(x)


def is_cosine_core(core):
    return core.func == cos and core.args and core.args[0].has(x)


def is_tangent_core(core):
    return core.func == tan and core.args and core.args[0].has(x)


def is_absolute_core(core):
    return core.func == Abs and core.args and core.args[0].has(x)


def is_radical_core(core):
    if isinstance(core, Pow):
        base, exponent = core.args
        if base.has(x) and exponent.is_Rational and exponent.q > 1:
            return True
    return False


def has_special_structure(expr_sym):
    if expr_sym.has(sin) or expr_sym.has(cos) or expr_sym.has(tan) or expr_sym.has(log) or expr_sym.has(Abs):
        return True

    for p in expr_sym.atoms(Pow):
        base, exponent = p.args

        if exponent.has(x) and not base.has(x):
            return True

        if base.has(x) and exponent.is_Rational and exponent.q > 1:
            return True

        if base.has(x) and exponent.has(x) and not exponent.is_Integer:
            return True

    return False


def classify_polynomial_or_rational(expr_sym):
    try:
        normalized = together(simplify(expr_sym))
        num, den = fraction(normalized)

        num_poly = Poly(num, x)
        den_poly = Poly(den, x)

        if den_poly.degree() > 0:
            return "RAT"

        deg = num_poly.degree()

        if deg == 0:
            return "CON"
        elif deg == 1:
            return "LIN"
        elif deg == 2:
            return "QUA"
        elif deg >= 3:
            return "POL"

    except Exception:
        pass

    return None


def classify_function_core(expr):
    expr_str = expr.strip()

    # --- Robust string-level LOG detection first ---
    # (runs after clean_expression; preprocess_for_sympy is called inside)
    if is_pure_log_expression_str(expr_str):
        return "LOG"

    try:
        expr_pre = preprocess_for_sympy(expr_str)

        expr_sym = parse_expr(
            expr_pre,
            transformations=transformations,
            local_dict={
                "x": x,
                "log": log,
                "Abs": Abs,
                "sin": sin,
                "cos": cos,
                "tan": tan,
                "sqrt": sqrt
            }
        )

        expr_sym = simplify(expr_sym)

        # 1. Constant
        if is_constant_function(expr_sym):
            return "CON"

        # 2. Special families first
        core_without_constants = remove_constant_addition(expr_sym)

        if core_without_constants != 0:
            coeff, core = split_scalar_and_core(core_without_constants)

            if not coeff.has(x):
                if is_exponential_core(core):
                    return "EXP"
                elif is_logarithmic_core(core):
                    return "LOG"
                elif is_sine_core(core):
                    return "SIN"
                elif is_cosine_core(core):
                    return "COS"
                elif is_tangent_core(core):
                    return "TAN"
                elif is_absolute_core(core):
                    return "ABS"
                elif is_radical_core(core):
                    return "RAD"

        # 3. Algebraic families after special families
        algebraic_family = classify_polynomial_or_rational(expr_sym)
        if algebraic_family is not None:
            return algebraic_family

        # 4. Mixed or complicated special structures
        if has_special_structure(expr_sym):
            return "OTF"

        return "OTF"

    except Exception:
        expr_fallback = expr_str.lower().replace(" ", "")

        # Strong log fallback first
        if is_pure_log_expression_str(expr_fallback):
            return "LOG"

        if re.fullmatch(r'[-+]?\d+(\.\d+)?', expr_fallback):
            return "CON"

        # trig
        if re.fullmatch(r'[-+]?\d*\.?\d*\*?sin\([^\)]+\)([-+].+)?', expr_fallback):
            return "SIN"
        if re.fullmatch(r'[-+]?\d*\.?\d*\*?cos\([^\)]+\)([-+].+)?', expr_fallback):
            return "COS"
        if re.fullmatch(r'[-+]?\d*\.?\d*\*?tan\([^\)]+\)([-+].+)?', expr_fallback):
            return "TAN"

        # absolute value
        if re.fullmatch(r'[-+]?\d*\.?\d*\*?abs\([^\)]+\)([-+].+)?', expr_fallback):
            return "ABS"

        # exponential fallback
        if re.fullmatch(
            r'[-+]?\d*\.?\d*\*?exp\(([-+]?\d*\.?\d*\*?x)?([-+]\d*\.?\d+)?\)([-+].+)?',
            expr_fallback
        ):
            return "EXP"

        if re.fullmatch(
            r'[-+]?\d*\.?\d*\*?[a-z0-9\.]+\^\(?[-+]?\d*\.?\d*\*?x([-+]\d*\.?\d+)?\)?([-+].+)?',
            expr_fallback
        ):
            return "EXP"

        # radical
        if "sqrt(" in expr_fallback:
            return "RAD"

        # rational
        if "/" in expr_fallback and "x" in expr_fallback:
            return "RAT"

        # polynomial fallback last
        if re.search(r'x(\*\*|\^)\(?2\)?', expr_fallback):
            return "QUA"
        elif re.search(r'x(\*\*|\^)[3-9]', expr_fallback):
            return "POL"
        elif "x" in expr_fallback and "/" not in expr_fallback:
            return "LIN"

        return "OTF"


def classify_function(expression):
    expr = clean_expression(expression)
    return classify_function_core(expr)


def classify_conic(matrix):
    A, B, C, D, E, F = matrix
    if B == 0 and A == C:
        return "conic_circle"
    elif B**2 - 4*A*C < 0:
        return "conic_ellipse"
    elif B**2 - 4*A*C == 0:
        return "conic_parabola"
    elif B**2 - 4*A*C > 0:
        return "conic_hyperbola"
    else:
        return "conic_other"


def parse_geogebra_file(file_path):
    with zipfile.ZipFile(file_path, 'r') as z:
        with z.open('geogebra.xml') as xml_file:
            tree = ET.parse(xml_file)
    return tree


# --- Main processing ---
all_object_types = set()
data = []

for filename in os.listdir(folder_path):
    if filename.endswith(".ggb"):
        file_path = os.path.join(folder_path, filename)
        try:
            tree = parse_geogebra_file(file_path)
            root = tree.getroot()

            counts = defaultdict(int)

            expressions = {
                exp.attrib["label"]: exp.attrib.get("exp", "")
                for exp in root.findall(".//expression[@type='function']")
            }

            total_functions = 0

            construction = root.find(".//construction")
            if construction is not None:
                elements = construction.findall("element")
                commands = construction.findall("command")

                num_elements = len(elements)
                element_labels = [el.attrib.get("label", "") for el in elements]

                command_outputs = set()
                for cmd in commands:
                    for out in cmd.findall("output"):
                        for k, v in out.attrib.items():
                            if k.startswith("a"):
                                command_outputs.add(v)

                free_labels = [lab for lab in element_labels if lab and lab not in command_outputs]
                num_steps = len(free_labels) + len(commands)
            else:
                num_elements = 0
                num_steps = 0

            counts["NS"] = num_steps
            counts["NE"] = num_elements
            all_object_types.add("NS")
            all_object_types.add("NE")

            for elem in root.findall(".//element"):
                obj_type = elem.attrib.get("type", "")
                label = elem.attrib.get("label", "")

                if obj_type == "function":
                    total_functions += 1
                    expr = expressions.get(label, "")
                    if expr:
                        ftype = classify_function(expr)
                        counts[ftype] += 1
                        all_object_types.add(ftype)
                    else:
                        counts["OTF"] += 1
                        all_object_types.add("OTF")

                elif obj_type == "parametric":
                    counts["PAR"] += 1
                    all_object_types.add("PAR")

                elif obj_type == "conic":
                    matrix_elem = elem.find(".//matrix")
                    if matrix_elem is not None:
                        try:
                            A = float(matrix_elem.attrib.get("A0", 0))
                            B = float(matrix_elem.attrib.get("A1", 0))
                            C = float(matrix_elem.attrib.get("A2", 0))
                            D = float(matrix_elem.attrib.get("A3", 0))
                            E = float(matrix_elem.attrib.get("A4", 0))
                            F = float(matrix_elem.attrib.get("A5", 0))
                            conic_type = classify_conic([A, B, C, D, E, F])
                            counts[conic_type] += 1
                            all_object_types.add(conic_type)
                        except Exception:
                            counts["conic_unknown"] += 1
                            all_object_types.add("conic_unknown")
                    else:
                        counts["conic_unknown"] += 1
                        all_object_types.add("conic_unknown")

                else:
                    counts[obj_type] += 1
                    all_object_types.add(obj_type)

            counts["TOT"] = total_functions
            all_object_types.add("TOT")

            row = {"filename": filename}
            row.update(counts)
            data.append(row)

        except Exception as e:
            print(f"Skipping {filename}: {e}")

df = pd.DataFrame(data)

for obj in all_object_types:
    if obj not in df.columns:
        df[obj] = 0

df = df.fillna(0)

function_order = [
    "CON", "LIN", "ABS", "QUA", "POL", "RAT", "RAD",
    "SIN", "COS", "TAN", "EXP", "LOG", "OTF",
    "TOT", "PAR", "NS", "NE"
]

remaining_types = sorted([obj for obj in all_object_types if obj not in function_order])
final_columns = ["filename"] + function_order + remaining_types

for col in final_columns:
    if col not in df.columns:
        df[col] = 0

df = df[final_columns]

print(df)
df.to_csv(output_file, index=False)
print(f"GeoGebra object tally saved to: {output_file}")

             filename  CON  LIN  ABS  QUA  POL  RAT  RAD  SIN  COS  TAN  EXP  \
0     sample4-tan.ggb    0    0    0    0    0    0    0    0    0  2.0  0.0   
1       sample3-2.ggb    0    0    0    0    0    0    0    0    0  0.0  0.0   
2  sample3-1-exp1.ggb    0    0    0    0    0    0    0    0    0  0.0  4.0   

   LOG  OTF  TOT  PAR  NS  NE  
0    0  1.0    3    0   3   3  
1    0  2.0    2    0   2   2  
2    0  0.0    4    0   4   4  
GeoGebra object tally saved to: /Users/guillermobautista/desktop/JKU/Function Arts/DataAnalysis/Sample3/sample3.csv


In [11]:
import os
import re
import zipfile
import xml.etree.ElementTree as ET
import pandas as pd
from collections import defaultdict

from sympy import (
    symbols, Poly, fraction, together, simplify, Pow, log,
    Add, Abs, sin, cos, tan, sqrt, Dummy
)
from sympy.parsing.sympy_parser import (
    parse_expr,
    standard_transformations,
    implicit_multiplication_application,
    convert_xor
)

# Folder path and output file
folder_path = "/Users/guillermobautista/desktop/JKU/Function Arts/DataAnalysis/Sample"
output_file = os.path.join(folder_path, "sample.csv")

x = symbols("x")
transformations = standard_transformations + (
    implicit_multiplication_application,
    convert_xor
)

# --- Helper functions ---

def extract_parenthesized(text, open_idx):
    """
    Given the index of '(' in text, return:
        (inside_text, closing_paren_index)
    """
    depth = 0
    for i in range(open_idx, len(text)):
        if text[i] == '(':
            depth += 1
        elif text[i] == ')':
            depth -= 1
            if depth == 0:
                return text[open_idx + 1:i], i
    raise ValueError("Unmatched parenthesis")


def split_top_level_args(s):
    """
    Split a string by commas at top level only.
    """
    parts = []
    depth = 0
    current = []

    for ch in s:
        if ch == '(':
            depth += 1
            current.append(ch)
        elif ch == ')':
            depth -= 1
            current.append(ch)
        elif ch == ',' and depth == 0:
            parts.append(''.join(current).strip())
            current = []
        else:
            current.append(ch)

    parts.append(''.join(current).strip())
    return parts


def normalize_inequality_symbols(s):
    return (
        s.replace("≤", "<=")
         .replace("≥", ">=")
         .replace("≦", "<=")
         .replace("≧", ">=")
    )


def is_domain_restriction(part):
    s = normalize_inequality_symbols(part.strip())

    if s.startswith("(") and s.endswith(")"):
        s = s[1:-1].strip()

    if "x" not in s:
        return False
    if not any(op in s for op in ["<", ">", "<=", ">="]):
        return False

    forbidden_formula_tokens = ["sin", "cos", "tan", "log", "ln", "lg", "sqrt", "^", "*", "/", "Abs", "abs"]
    if any(tok in s for tok in forbidden_formula_tokens):
        return False

    chained_patterns = [
        r'^.+?(<=|<)\s*x\s*(<=|<)\s*.+$',
        r'^.+?(>=|>)\s*x\s*(>=|>)\s*.+$',
        r'^x\s*(<=|<)\s*.+$',
        r'^x\s*(>=|>)\s*.+$',
        r'^.+\s*(<=|<)\s*x$',
        r'^.+\s*(>=|>)\s*x$'
    ]

    return any(re.fullmatch(p, s) for p in chained_patterns)


def strip_trailing_domain_restriction(expr):
    parts = split_top_level_args(expr)
    if len(parts) >= 2:
        last_part = parts[-1].strip()
        if is_domain_restriction(last_part):
            return ','.join(parts[:-1]).strip()
    return expr.strip()


def strip_boolean_interval_factor(expr):
    s = expr.strip()

    changed = True
    while changed and s.startswith("(") and s.endswith(")"):
        changed = False
        depth = 0
        wraps_all = True
        for i, ch in enumerate(s):
            if ch == "(":
                depth += 1
            elif ch == ")":
                depth -= 1
                if depth == 0 and i != len(s) - 1:
                    wraps_all = False
                    break
        if wraps_all:
            s = s[1:-1].strip()
            changed = True

    factors = []
    depth = 0
    current = []
    for ch in s:
        if ch == "(":
            depth += 1
            current.append(ch)
        elif ch == ")":
            depth -= 1
            current.append(ch)
        elif ch == "*" and depth == 0:
            factors.append(''.join(current).strip())
            current = []
        else:
            current.append(ch)
    factors.append(''.join(current).strip())

    if len(factors) != 2:
        return expr.strip()

    f1, f2 = factors[0], factors[1]

    if is_domain_restriction(f1):
        return f2
    if is_domain_restriction(f2):
        return f1

    return expr.strip()


def clean_expression(expr):
    expr = expr.strip()

    if ":" in expr:
        left, right = expr.split(":", 1)
        if "=" in right:
            expr = right.strip()

    match = re.match(r'\s*\w+\(x\)\s*=\s*If\[[^\]]+\s*,\s*(.+?)\s*\]\s*$', expr)
    if match:
        rhs = match.group(1).strip()
        rhs = strip_trailing_domain_restriction(rhs)
        rhs = strip_boolean_interval_factor(rhs)
        return rhs

    match = re.match(r'\s*\w+\(x\)\s*=\s*(.+)$', expr)
    if match:
        rhs = match.group(1).strip()
        rhs = strip_trailing_domain_restriction(rhs)
        rhs = strip_boolean_interval_factor(rhs)
        return rhs

    match = re.match(r'\s*y\s*=\s*(.+)$', expr, flags=re.IGNORECASE)
    if match:
        rhs = match.group(1).strip()
        rhs = strip_trailing_domain_restriction(rhs)
        rhs = strip_boolean_interval_factor(rhs)
        return rhs

    if "=" in expr:
        parts = expr.split("=", 1)
        rhs = parts[1].strip()
        rhs = strip_trailing_domain_restriction(rhs)
        rhs = strip_boolean_interval_factor(rhs)
        return rhs

    expr = strip_trailing_domain_restriction(expr)
    expr = strip_boolean_interval_factor(expr)
    return expr


def replace_log_with_base_underscore(expr):
    """
    Convert:
        log_10(x)     -> log(x,10)
        log_{10}(x)   -> log(x,10)
        log_(10)(x)   -> log(x,10)
    """
    pattern1 = re.compile(r'log_\{\s*([^{}]+?)\s*\}\s*\(')
    pattern2 = re.compile(r'log_([A-Za-z0-9\.\+\-\*/]+)\s*\(')
    pattern3 = re.compile(r'log_\(\s*([^\(\)]+?)\s*\)\s*\(')

    while True:
        m = pattern1.search(expr)
        if not m:
            break
        base = m.group(1).strip()
        start = m.start()
        open_paren = m.end() - 1
        arg, close_idx = extract_parenthesized(expr, open_paren)
        replacement = f'log({arg},{base})'
        expr = expr[:start] + replacement + expr[close_idx + 1:]

    while True:
        m = pattern3.search(expr)
        if not m:
            break
        base = m.group(1).strip()
        start = m.start()
        open_paren = m.end() - 1
        arg, close_idx = extract_parenthesized(expr, open_paren)
        replacement = f'log({arg},{base})'
        expr = expr[:start] + replacement + expr[close_idx + 1:]

    while True:
        m = pattern2.search(expr)
        if not m:
            break
        base = m.group(1).strip()
        start = m.start()
        open_paren = m.end() - 1
        arg, close_idx = extract_parenthesized(expr, open_paren)
        replacement = f'log({arg},{base})'
        expr = expr[:start] + replacement + expr[close_idx + 1:]

    return expr


def looks_like_numeric_constant(s):
    s = s.strip()
    if not s:
        return False
    if "x" in s.lower():
        return False
    return bool(re.fullmatch(r'[-+]?\d*\.?\d+([eE][-+]?\d+)?', s))


def replace_log_two_argument_form(expr):
    """
    Convert only GeoGebra-style two-argument log:
        log(10, x)   -> log(x,10)
        log(2, x+1)  -> log(x+1,2)

    Leave canonical SymPy-style forms unchanged:
        log(x,10)
    """
    i = 0
    result = []

    while i < len(expr):
        if expr[i:i+4].lower() == 'log(':
            open_idx = i + 3
            inside, close_idx = extract_parenthesized(expr, open_idx)
            parts = split_top_level_args(inside)

            if len(parts) == 2:
                first, second = parts[0].strip(), parts[1].strip()

                if looks_like_numeric_constant(first) and "x" in second.lower():
                    result.append(f'log({second},{first})')
                else:
                    result.append(f'log({inside})')
            else:
                result.append(f'log({inside})')

            i = close_idx + 1
        else:
            result.append(expr[i])
            i += 1

    return ''.join(result)


def preprocess_for_sympy(expr):
    expr = expr.strip()

    # FIX: convert GeoGebra's lg( (log base 10) to log( before any other processing
    expr = re.sub(r'\blg\s*\(', 'log(', expr, flags=re.IGNORECASE)

    expr = re.sub(r'\bln\s*\(', 'log(', expr, flags=re.IGNORECASE)
    expr = re.sub(r'\babs\s*\(', 'Abs(', expr, flags=re.IGNORECASE)

    expr = replace_log_with_base_underscore(expr)
    expr = replace_log_two_argument_form(expr)

    return expr


def replace_log_calls_with_placeholder(expr, placeholder="L"):
    """
    Replace every full log(...) call (with balanced parentheses) by a placeholder.
    Works on preprocessed strings where log notation has already been normalized.
    """
    result = []
    i = 0

    while i < len(expr):
        if expr[i:i+4].lower() == 'log(':
            open_idx = i + 3
            _, close_idx = extract_parenthesized(expr, open_idx)
            result.append(placeholder)
            i = close_idx + 1
        else:
            result.append(expr[i])
            i += 1

    return ''.join(result)


def replace_trig_calls_with_placeholder(expr, func_name, placeholder="T"):
    """
    Replace every full trig call func_name(...) by a placeholder.
    Example:
        func_name='tan'
        (tan(x)+1)^2  -> (T+1)^2
    """
    result = []
    i = 0
    token = f"{func_name}("

    while i < len(expr):
        if expr[i:i+len(token)].lower() == token:
            open_idx = i + len(func_name)
            _, close_idx = extract_parenthesized(expr, open_idx)
            result.append(placeholder)
            i = close_idx + 1
        else:
            result.append(expr[i])
            i += 1

    return ''.join(result)


def is_pure_single_trig_expression_str(expr, trig_name):
    """
    Detect expressions built only from one trig family plus constants/operators.

    Accepts:
        sin(x)
        (sin(x)+1)^2
        2*cos(x)-3
        tan(x)^2 + 2*tan(x) + 1

    Rejects:
        x + sin(x)
        x*sin(x)
        sin(x) + cos(x)
        tan(x) + log(x)
    """
    expr_pre = preprocess_for_sympy(expr)
    expr_pre = expr_pre.replace(" ", "")
    lower = expr_pre.lower()

    target = f"{trig_name}("
    if target not in lower:
        return False

    # Reject presence of other special families
    other_trigs = {"sin", "cos", "tan"} - {trig_name}
    forbidden = [f"{name}(" for name in other_trigs] + ["log(", "sqrt(", "abs(", "exp("]
    if any(tok in lower for tok in forbidden):
        return False

    replaced = replace_trig_calls_with_placeholder(expr_pre, trig_name, "T")

    # Remove numeric constants, placeholder, operators, parentheses, commas, powers
    stripped = re.sub(r'[-+]?\d*\.?\d+([eE][-+]?\d+)?', '', replaced)
    stripped = stripped.replace("T", "")
    stripped = re.sub(r'[\+\-\*/\^\(\),]', '', stripped)

    # If x remains outside trig blocks, do not classify as pure trig family
    return "x" not in stripped.lower()


def is_pure_log_expression_str(expr):
    """
    String-level detector for logarithmic expressions.
    After preprocessing (which converts lg, ln -> log), replace every log(...)
    block by a placeholder L. If no raw x remains outside those blocks,
    classify as LOG.

    Accepts:
        log_10(x)
        (log_10(x+2))^2
        (log_10(2x-6))^3
        ln(x)
        ln(x)^2
        lg(x)
        (lg(x+2))^2
        (lg(x+3))^3

    Rejects:
        x + log_10(x)
        x*log_10(x)
        log_10(x)/x
    """
    # Preprocess first so lg( and ln( become log(
    expr_pre = preprocess_for_sympy(expr)
    expr_pre = expr_pre.replace(" ", "")

    # If there is no log at all, it is not logarithmic
    if "log(" not in expr_pre.lower():
        return False

    # Exclude other special families for plain LOG classification
    lower = expr_pre.lower()
    forbidden = ["sin(", "cos(", "tan(", "sqrt(", "abs(", "exp("]
    if any(tok in lower for tok in forbidden):
        return False

    replaced = replace_log_calls_with_placeholder(expr_pre, "L")

    # Remove numeric constants, operators, placeholders, parentheses, commas, powers
    stripped = re.sub(r'[-+]?\d*\.?\d+([eE][-+]?\d+)?', '', replaced)
    stripped = stripped.replace("L", "")
    stripped = re.sub(r'[\+\-\*/\^\(\),]', '', stripped)

    # If x remains outside the log blocks, it is not a pure log family
    return "x" not in stripped.lower()


def remove_constant_addition(expr_sym):
    var_terms = [t for t in Add.make_args(expr_sym) if t.has(x)]
    if not var_terms:
        return 0
    return simplify(Add(*var_terms))


def split_scalar_and_core(expr_sym):
    coeff, rest = expr_sym.as_independent(x, as_Add=False)
    return simplify(coeff), simplify(rest)


# --- Family detectors ---

def is_constant_function(expr_sym):
    return not expr_sym.has(x)


def is_affine_in_x(expr):
    try:
        expr = simplify(expr)
        poly = Poly(expr, x)
        return poly.degree() <= 1
    except Exception:
        return False


def is_exponential_core(core):
    rewritten = simplify(core.rewrite(Pow))

    if isinstance(rewritten, Pow):
        base, exponent = rewritten.args

        if base.has(x):
            return False
        if not exponent.has(x):
            return False
        if not is_affine_in_x(exponent):
            return False

        return True

    return False


def is_logarithmic_core(core):
    """
    SymPy-level detector for logarithmic expressions.

    Handles:
        log(x+10)        — plain log
        log(x+10)**2     — Pow whose base is a log (e.g. from (lg(x+2))^2)
        log(x+10)**3     — higher integer powers of log
        2*log(x+10)      — scalar multiple (split_scalar_and_core handles upstream)
    """
    core = simplify(core)

    if core.has(sin) or core.has(cos) or core.has(tan) or core.has(Abs):
        return False

    # Handle Pow(log(...), n) — e.g. log(x+2)**2 or log(x+3)**3
    if isinstance(core, Pow):
        base, exponent = core.args
        if not exponent.has(x) and isinstance(base, log) and base.args[0].has(x):
            return True

    log_atoms = [a for a in core.atoms(log) if len(a.args) >= 1 and a.args[0].has(x)]

    if not log_atoms:
        return False

    replacements = {}
    for i, atom in enumerate(log_atoms):
        replacements[atom] = Dummy(f"L{i}")
    replaced = core.xreplace(replacements)

    return not replaced.has(x)


def is_sine_core(core):
    core = simplify(core)

    if core.has(cos) or core.has(tan) or core.has(log) or core.has(Abs):
        return False

    sin_atoms = [a for a in core.atoms(sin) if a.args and a.args[0].has(x)]
    if not sin_atoms:
        return False

    replacements = {}
    for i, atom in enumerate(sin_atoms):
        replacements[atom] = Dummy(f"S{i}")
    replaced = simplify(core.xreplace(replacements))

    return not replaced.has(x)


def is_cosine_core(core):
    core = simplify(core)

    if core.has(sin) or core.has(tan) or core.has(log) or core.has(Abs):
        return False

    cos_atoms = [a for a in core.atoms(cos) if a.args and a.args[0].has(x)]
    if not cos_atoms:
        return False

    replacements = {}
    for i, atom in enumerate(cos_atoms):
        replacements[atom] = Dummy(f"C{i}")
    replaced = simplify(core.xreplace(replacements))

    return not replaced.has(x)


def is_tangent_core(core):
    core = simplify(core)

    if core.has(sin) or core.has(cos) or core.has(log) or core.has(Abs):
        return False

    tan_atoms = [a for a in core.atoms(tan) if a.args and a.args[0].has(x)]
    if not tan_atoms:
        return False

    replacements = {}
    for i, atom in enumerate(tan_atoms):
        replacements[atom] = Dummy(f"T{i}")
    replaced = simplify(core.xreplace(replacements))

    return not replaced.has(x)


def is_absolute_core(core):
    return core.func == Abs and core.args and core.args[0].has(x)


def is_radical_core(core):
    if isinstance(core, Pow):
        base, exponent = core.args
        if base.has(x) and exponent.is_Rational and exponent.q > 1:
            return True
    return False


def has_special_structure(expr_sym):
    if expr_sym.has(sin) or expr_sym.has(cos) or expr_sym.has(tan) or expr_sym.has(log) or expr_sym.has(Abs):
        return True

    for p in expr_sym.atoms(Pow):
        base, exponent = p.args

        if exponent.has(x) and not base.has(x):
            return True

        if base.has(x) and exponent.is_Rational and exponent.q > 1:
            return True

        if base.has(x) and exponent.has(x) and not exponent.is_Integer:
            return True

    return False


def classify_polynomial_or_rational(expr_sym):
    try:
        normalized = together(simplify(expr_sym))
        num, den = fraction(normalized)

        num_poly = Poly(num, x)
        den_poly = Poly(den, x)

        if den_poly.degree() > 0:
            return "RAT"

        deg = num_poly.degree()

        if deg == 0:
            return "CON"
        elif deg == 1:
            return "LIN"
        elif deg == 2:
            return "QUA"
        elif deg >= 3:
            return "POL"

    except Exception:
        pass

    return None


def classify_function_core(expr):
    expr_str = expr.strip()

    # --- Robust string-level special-family detection first ---
    if is_pure_log_expression_str(expr_str):
        return "LOG"
    if is_pure_single_trig_expression_str(expr_str, "sin"):
        return "SIN"
    if is_pure_single_trig_expression_str(expr_str, "cos"):
        return "COS"
    if is_pure_single_trig_expression_str(expr_str, "tan"):
        return "TAN"

    try:
        expr_pre = preprocess_for_sympy(expr_str)

        expr_sym = parse_expr(
            expr_pre,
            transformations=transformations,
            local_dict={
                "x": x,
                "log": log,
                "Abs": Abs,
                "sin": sin,
                "cos": cos,
                "tan": tan,
                "sqrt": sqrt
            }
        )

        expr_sym = simplify(expr_sym)

        # 1. Constant
        if is_constant_function(expr_sym):
            return "CON"

        # 2. Special families first
        core_without_constants = remove_constant_addition(expr_sym)

        if core_without_constants != 0:
            coeff, core = split_scalar_and_core(core_without_constants)

            if not coeff.has(x):
                if is_exponential_core(core):
                    return "EXP"
                elif is_logarithmic_core(core):
                    return "LOG"
                elif is_sine_core(core):
                    return "SIN"
                elif is_cosine_core(core):
                    return "COS"
                elif is_tangent_core(core):
                    return "TAN"
                elif is_absolute_core(core):
                    return "ABS"
                elif is_radical_core(core):
                    return "RAD"

        # 3. Algebraic families after special families
        algebraic_family = classify_polynomial_or_rational(expr_sym)
        if algebraic_family is not None:
            return algebraic_family

        # 4. Mixed or complicated special structures
        if has_special_structure(expr_sym):
            return "OTF"

        return "OTF"

    except Exception:
        expr_fallback = expr_str.lower().replace(" ", "")

        # Strong special-family fallbacks first
        if is_pure_log_expression_str(expr_fallback):
            return "LOG"
        if is_pure_single_trig_expression_str(expr_fallback, "sin"):
            return "SIN"
        if is_pure_single_trig_expression_str(expr_fallback, "cos"):
            return "COS"
        if is_pure_single_trig_expression_str(expr_fallback, "tan"):
            return "TAN"

        if re.fullmatch(r'[-+]?\d+(\.\d+)?', expr_fallback):
            return "CON"

        # trig
        if re.fullmatch(r'[-+]?\d*\.?\d*\*?sin\([^\)]+\)([-+].+)?', expr_fallback):
            return "SIN"
        if re.fullmatch(r'[-+]?\d*\.?\d*\*?cos\([^\)]+\)([-+].+)?', expr_fallback):
            return "COS"
        if re.fullmatch(r'[-+]?\d*\.?\d*\*?tan\([^\)]+\)([-+].+)?', expr_fallback):
            return "TAN"

        # absolute value
        if re.fullmatch(r'[-+]?\d*\.?\d*\*?abs\([^\)]+\)([-+].+)?', expr_fallback):
            return "ABS"

        # exponential fallback
        if re.fullmatch(
            r'[-+]?\d*\.?\d*\*?exp\(([-+]?\d*\.?\d*\*?x)?([-+]\d*\.?\d+)?\)([-+].+)?',
            expr_fallback
        ):
            return "EXP"

        if re.fullmatch(
            r'[-+]?\d*\.?\d*\*?[a-z0-9\.]+\^\(?[-+]?\d*\.?\d*\*?x([-+]\d*\.?\d+)?\)?([-+].+)?',
            expr_fallback
        ):
            return "EXP"

        # radical
        if "sqrt(" in expr_fallback:
            return "RAD"

        # rational
        if "/" in expr_fallback and "x" in expr_fallback:
            return "RAT"

        # polynomial fallback last
        if re.search(r'x(\*\*|\^)\(?2\)?', expr_fallback):
            return "QUA"
        elif re.search(r'x(\*\*|\^)[3-9]', expr_fallback):
            return "POL"
        elif "x" in expr_fallback and "/" not in expr_fallback:
            return "LIN"

        return "OTF"


def classify_function(expression):
    expr = clean_expression(expression)
    return classify_function_core(expr)


def classify_conic(matrix):
    A, B, C, D, E, F = matrix
    if B == 0 and A == C:
        return "conic_circle"
    elif B**2 - 4*A*C < 0:
        return "conic_ellipse"
    elif B**2 - 4*A*C == 0:
        return "conic_parabola"
    elif B**2 - 4*A*C > 0:
        return "conic_hyperbola"
    else:
        return "conic_other"


def parse_geogebra_file(file_path):
    with zipfile.ZipFile(file_path, 'r') as z:
        with z.open('geogebra.xml') as xml_file:
            tree = ET.parse(xml_file)
    return tree


# --- Main processing ---
all_object_types = set()
data = []

for filename in os.listdir(folder_path):
    if filename.endswith(".ggb"):
        file_path = os.path.join(folder_path, filename)
        try:
            tree = parse_geogebra_file(file_path)
            root = tree.getroot()

            counts = defaultdict(int)

            expressions = {
                exp.attrib["label"]: exp.attrib.get("exp", "")
                for exp in root.findall(".//expression[@type='function']")
            }

            total_functions = 0

            construction = root.find(".//construction")
            if construction is not None:
                elements = construction.findall("element")
                commands = construction.findall("command")

                num_elements = len(elements)
                element_labels = [el.attrib.get("label", "") for el in elements]

                command_outputs = set()
                for cmd in commands:
                    for out in cmd.findall("output"):
                        for k, v in out.attrib.items():
                            if k.startswith("a"):
                                command_outputs.add(v)

                free_labels = [lab for lab in element_labels if lab and lab not in command_outputs]
                num_steps = len(free_labels) + len(commands)
            else:
                num_elements = 0
                num_steps = 0

            counts["NS"] = num_steps
            counts["NE"] = num_elements
            all_object_types.add("NS")
            all_object_types.add("NE")

            for elem in root.findall(".//element"):
                obj_type = elem.attrib.get("type", "")
                label = elem.attrib.get("label", "")

                if obj_type == "function":
                    total_functions += 1
                    expr = expressions.get(label, "")
                    if expr:
                        ftype = classify_function(expr)
                        counts[ftype] += 1
                        all_object_types.add(ftype)
                    else:
                        counts["OTF"] += 1
                        all_object_types.add("OTF")

                elif obj_type == "parametric":
                    counts["PAR"] += 1
                    all_object_types.add("PAR")

                elif obj_type == "conic":
                    matrix_elem = elem.find(".//matrix")
                    if matrix_elem is not None:
                        try:
                            A = float(matrix_elem.attrib.get("A0", 0))
                            B = float(matrix_elem.attrib.get("A1", 0))
                            C = float(matrix_elem.attrib.get("A2", 0))
                            D = float(matrix_elem.attrib.get("A3", 0))
                            E = float(matrix_elem.attrib.get("A4", 0))
                            F = float(matrix_elem.attrib.get("A5", 0))
                            conic_type = classify_conic([A, B, C, D, E, F])
                            counts[conic_type] += 1
                            all_object_types.add(conic_type)
                        except Exception:
                            counts["conic_unknown"] += 1
                            all_object_types.add("conic_unknown")
                    else:
                        counts["conic_unknown"] += 1
                        all_object_types.add("conic_unknown")

                else:
                    counts[obj_type] += 1
                    all_object_types.add(obj_type)

            counts["TOT"] = total_functions
            all_object_types.add("TOT")

            row = {"filename": filename}
            row.update(counts)
            data.append(row)

        except Exception as e:
            print(f"Skipping {filename}: {e}")

df = pd.DataFrame(data)

for obj in all_object_types:
    if obj not in df.columns:
        df[obj] = 0

df = df.fillna(0)

function_order = [
    "CON", "LIN", "ABS", "QUA", "POL", "RAT", "RAD",
    "SIN", "COS", "TAN", "EXP", "LOG", "OTF",
    "TOT", "PAR", "NS", "NE"
]

remaining_types = sorted([obj for obj in all_object_types if obj not in function_order])
final_columns = ["filename"] + function_order + remaining_types

for col in final_columns:
    if col not in df.columns:
        df[col] = 0

df = df[final_columns]

print(df)
df.to_csv(output_file, index=False)
print(f"GeoGebra object tally saved to: {output_file}")

        filename  CON    LIN  ABS   QUA  POL   RAT  RAD  SIN   COS  ...  TOT  \
0    sample9.ggb  0.0    0.0    0   0.0  0.0   1.0  1.0  0.0   0.0  ...    3   
1    sample8.ggb  0.0    1.0    0   0.0  0.0   1.0  0.0  0.0   0.0  ...    4   
2   sample10.ggb  0.0    0.0    0   0.0  0.0   0.0  0.0  0.0   0.0  ...    1   
3   sample11.ggb  0.0    0.0    0   0.0  0.0   0.0  0.0  0.0   0.0  ...    3   
4   sample12.ggb  0.0    0.0    0   0.0  0.0   0.0  0.0  0.0   0.0  ...    1   
5    sample3.ggb  0.0    0.0    0   0.0  0.0   1.0  1.0  1.0   0.0  ...    5   
6    sample2.ggb  0.0    0.0    0   0.0  0.0   1.0  1.0  0.0   0.0  ...    3   
7    sample1.ggb  2.0    4.0    0   4.0  2.0   0.0  0.0  1.0   0.0  ...   13   
8    sample5.ggb  0.0    0.0    0   5.0  0.0   0.0  0.0  0.0  10.0  ...   15   
9    sample4.ggb  0.0    1.0    0   0.0  0.0   1.0  0.0  0.0   1.0  ...    6   
10   sample6.ggb  0.0  378.0    0   0.0  0.0   4.0  0.0  0.0   0.0  ...  382   
11   sample7.ggb  1.0   38.0    0  22.0 